In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------
# Load and prepare data
# ----------------------
wiki_results = pd.read_csv("results_wikipedia.csv")
cons_results = pd.read_csv("results_conspiracy.csv")

# Keep only clean and meani
wiki_results = wiki_results[wiki_results['context_type'].isin(['clean', 'meani'])].copy()
cons_results = cons_results[cons_results['context_type'].isin(['clean', 'meani'])].copy()

# Label content type
wiki_results['content'] = 'wikipedia'
cons_results['content'] = 'conspiracy'

# Combine datasets
data = pd.concat([wiki_results, cons_results], ignore_index=True)

# Convert to categorical with proper ordering
data['content'] = data['content'].astype('category').cat.set_categories(['wikipedia', 'conspiracy'], ordered=True)
data['context_type'] = data['context_type'].astype('category').cat.set_categories(['clean', 'meani'], ordered=True)

# ----------------------
# Fit LMM (random intercepts only)
# ----------------------
model_lmm = smf.mixedlm(
    "accuracy ~ content * context_type + scale(text_length)",
    data,
    groups="model_name"
)

result_lmm = model_lmm.fit(reml=True, method="lbfgs")
print(result_lmm.summary())

# ----------------------
# Compute residuals
# ----------------------
data['fitted'] = result_lmm.fittedvalues
data['residuals'] = data['accuracy'] - data['fitted']

# ----------------------
# Plot histogram of residuals
# ----------------------
plt.figure(figsize=(8,5))
sns.histplot(data['residuals'], bins=50, kde=True, color='skyblue')
plt.axvline(0, color='red', linestyle='--')
plt.title("Histogram of LMM Residuals (accuracy ~ content * context_type)")
plt.xlabel("Residual")
plt.ylabel("Count")
plt.show()

# ----------------------
# Plot residuals vs fitted values
# ----------------------
plt.figure(figsize=(8,5))
plt.scatter(data['fitted'], data['residuals'], alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.title("Residuals vs Fitted Values")
plt.xlabel("Fitted accuracy")
plt.ylabel("Residual")
plt.show()

# ----------------------
# Optional: check for extreme values
# ----------------------
extremes = data[(data['accuracy'] < 0.01) | (data['accuracy'] > 0.99)]
print(f"Number of extreme accuracy values (<0.01 or >0.99): {len(extremes)}")
print(extremes[['model_name','content','context_type','accuracy']])


NameError: name 'result_lmm' is not defined

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

# wiki vs conspiracy, clean and meani, original and original + random slopes
############################################################################

wiki_results = pd.read_csv("results_wikipedia.csv")
cons_results = pd.read_csv("results_conspiracy.csv")

wiki_results = wiki_results[wiki_results['context_type'].isin(['clean', 'meani'])].copy()
cons_results = cons_results[cons_results['context_type'].isin(['clean', 'meani'])].copy()

wiki_results['content'] = 'wikipedia'
cons_results['content'] = 'conspiracy'

data = pd.concat([wiki_results, cons_results], ignore_index=True)

data['content'] = (
    data['content']
    .astype('category')
    .cat.set_categories(['wikipedia', 'conspiracy'], ordered=True)
)

data['context_type'] = (
    data['context_type']
    .astype('category')
    .cat.set_categories(['clean', 'meani'], ordered=True)
)

data['text_length_s'] = (data['text_length'] - data['text_length'].mean()) / data['text_length'].std()

############################################

model_original = smf.mixedlm(
    "accuracy ~ content * context_type + text_length_s",
    data,
    groups="model_name",
    vc_formula={"context": "0 + C(context_title)"}
)

result_original = model_original.fit(reml=True, method="lbfgs")

print("random intercepts only:")
print(result_original.summary())

#########################################

# model_slope = smf.mixedlm(
#     "accuracy ~ content * context_type + text_length_s",
#     data,
#     groups="model_name",
#     re_formula="~ content",
#     vc_formula={"context": "0 + C(context_title)"}
# )

# result_slope = model_slope.fit(reml=True, method="lbfgs")
# print("random intercepts + random slopes:")
# print(result_slope.summary())



In [1]:
import pandas as pd
import statsmodels.formula.api as smf

# wiki vs conspiracy, clean and meani, Qwen only
#########################################################################

wiki_results = pd.read_csv("results_wikipedia.csv")
cons_results = pd.read_csv("results_conspiracy.csv")

wiki_results = wiki_results[wiki_results['context_type'].isin(['clean', 'meani'])].copy()
cons_results = cons_results[cons_results['context_type'].isin(['clean', 'meani'])].copy()

wiki_results['content'] = 'wikipedia'
cons_results['content'] = 'conspiracy'

data = pd.concat([wiki_results, cons_results], ignore_index=True)
data = data[data["model_name"] == "Qwen/Qwen3-8B"].copy()

data['content'] = (
    data['content']
    .astype('category')
    .cat.set_categories(['wikipedia', 'conspiracy'], ordered=True)
)

data['context_type'] = (
    data['context_type']
    .astype('category')
    .cat.set_categories(['clean', 'meani'], ordered=True)
)

# standardize within Qwen
data['text_length_s'] = (
    (data['text_length'] - data['text_length'].mean())
    / data['text_length'].std()
)

########################################
model_qwen = smf.mixedlm(
    "accuracy ~ content * context_type + text_length_s",
    data,
    groups="context_title"
)

result_qwen = model_qwen.fit(reml=True, method="lbfgs")

print("Qwen only:")
print(result_qwen.summary())


/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


Qwen only:
                         Mixed Linear Model Regression Results
Model:                        MixedLM           Dependent Variable:           accuracy 
No. Observations:             4008              Method:                       REML     
No. Groups:                   2004              Scale:                        0.0003   
Min. group size:              2                 Log-Likelihood:               9562.7228
Max. group size:              2                 Converged:                    Yes      
Mean group size:              2.0                                                      
---------------------------------------------------------------------------------------
                                            Coef.  Std.Err.    z    P>|z| [0.025 0.975]
---------------------------------------------------------------------------------------
Intercept                                    0.563    0.001 749.150 0.000  0.562  0.565
content[T.conspiracy]                        0

In [10]:
import pandas as pd
import statsmodels.formula.api as smf

# wiki vs conspiracy, clean and meani, gemma only
#########################################################################

wiki_results = pd.read_csv("results_wikipedia.csv")
cons_results = pd.read_csv("results_conspiracy.csv")

wiki_results = wiki_results[wiki_results['context_type'].isin(['clean', 'meani'])].copy()
cons_results = cons_results[cons_results['context_type'].isin(['clean', 'meani'])].copy()

wiki_results['content'] = 'wikipedia'
cons_results['content'] = 'conspiracy'

data = pd.concat([wiki_results, cons_results], ignore_index=True)
data = data[data["model_name"] == "google/gemma-2-9b"].copy()
print(len(data))

data['content'] = (
    data['content']
    .astype('category')
    .cat.set_categories(['wikipedia', 'conspiracy'], ordered=True)
)

data['context_type'] = (
    data['context_type']
    .astype('category')
    .cat.set_categories(['clean', 'meani'], ordered=True)
)

# standardize within Qwen
data['text_length_s'] = (
    (data['text_length'] - data['text_length'].mean())
    / data['text_length'].std()
)

########################################
model_qwen = smf.mixedlm(
    "accuracy ~ content * context_type + text_length_s",
    data,
    groups="context_title"
)

result_qwen = model_qwen.fit(reml=True, method="lbfgs")

print("Qwen only:")
print(result_qwen.summary())


4008


/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


Qwen only:
                         Mixed Linear Model Regression Results
Model:                        MixedLM           Dependent Variable:           accuracy 
No. Observations:             4008              Method:                       REML     
No. Groups:                   2004              Scale:                        0.0019   
Min. group size:              2                 Log-Likelihood:               5273.8183
Max. group size:              2                 Converged:                    Yes      
Mean group size:              2.0                                                      
---------------------------------------------------------------------------------------
                                            Coef.  Std.Err.    z    P>|z| [0.025 0.975]
---------------------------------------------------------------------------------------
Intercept                                    0.414    0.002 176.411 0.000  0.409  0.418
content[T.conspiracy]                        0

In [9]:
import pandas as pd
import statsmodels.formula.api as smf

# wiki vs conspiracy, clean and meani, mistral only
#########################################################################

wiki_results = pd.read_csv("results_wikipedia.csv")
cons_results = pd.read_csv("results_conspiracy.csv")

wiki_results = wiki_results[wiki_results['context_type'].isin(['clean', 'meani'])].copy()
cons_results = cons_results[cons_results['context_type'].isin(['clean', 'meani'])].copy()

wiki_results['content'] = 'wikipedia'
cons_results['content'] = 'conspiracy'

data = pd.concat([wiki_results, cons_results], ignore_index=True)
data = data[data["model_name"] == "mistralai/Mistral-7B-Instruct-v0.3"].copy()
print(len(data))

data['content'] = (
    data['content']
    .astype('category')
    .cat.set_categories(['wikipedia', 'conspiracy'], ordered=True)
)

data['context_type'] = (
    data['context_type']
    .astype('category')
    .cat.set_categories(['clean', 'meani'], ordered=True)
)

# standardize within Qwen
data['text_length_s'] = (
    (data['text_length'] - data['text_length'].mean())
    / data['text_length'].std()
)

########################################
model_qwen = smf.mixedlm(
    "accuracy ~ content * context_type + text_length_s",
    data,
    groups="context_title"
)

result_qwen = model_qwen.fit(reml=True, method="lbfgs")

print("Qwen only:")
print(result_qwen.summary())


4008


/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)


LinAlgError: Singular matrix

In [11]:
import pandas as pd
import statsmodels.formula.api as smf

# wiki vs conspiracy, clean and meani, llama only
#########################################################################

wiki_results = pd.read_csv("results_wikipedia.csv")
cons_results = pd.read_csv("results_conspiracy.csv")

wiki_results = wiki_results[wiki_results['context_type'].isin(['clean', 'meani'])].copy()
cons_results = cons_results[cons_results['context_type'].isin(['clean', 'meani'])].copy()

wiki_results['content'] = 'wikipedia'
cons_results['content'] = 'conspiracy'

data = pd.concat([wiki_results, cons_results], ignore_index=True)
subset = data[data['model_name'] == "llama"]
print(subset.groupby(['context_title', 'content', 'context_type']).size())
data = data[data["model_name"] == "meta-llama/Llama-3.1-8B"].copy()



data['content'] = (
    data['content']
    .astype('category')
    .cat.set_categories(['wikipedia', 'conspiracy'], ordered=True)
)

data['context_type'] = (
    data['context_type']
    .astype('category')
    .cat.set_categories(['clean', 'meani'], ordered=True)
)

# standardize within Qwen
data['text_length_s'] = (
    (data['text_length'] - data['text_length'].mean())
    / data['text_length'].std()
)

########################################
model_qwen = smf.mixedlm(
    "accuracy ~ content * context_type + text_length_s",
    data,
    groups="context_title"
)

result_qwen = model_qwen.fit(reml=True, method="lbfgs")

print("Qwen only:")
print(result_qwen.summary())


Series([], dtype: int64)


/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)


LinAlgError: Singular matrix

In [20]:
import pandas as pd

# Load Wikipedia CSV
df = pd.read_csv("results_wikipedia.csv")

# Group by model_name and get unique context_titles
for model, group in df.groupby("model_name"):
    unique_titles = group["context_title"].unique()
    print(f"\nModel: {model}")
    for title in unique_titles:
        print(f"  - {title}")




Model: Qwen/Qwen3-8B
  - 10,000_BC_(TV_series)
  - Kannada_(Unicode_block)
  - 128_Lit
  - Karatia_Union
  - 131st_Reconnaissance_Battalion_(Ukraine)
  - Karl_Höltermann
  - 1660_Wood
  - Kashima_District,_Ishikawa
  - 1895_Tulane_Olive_and_Blue_football_team
  - Kashtabhanga_Union
  - 1905_Nebraska_Cornhuskers_football_team
  - Kate_Burton_(actress)
  - 1907_West_Down_by-election
  - Katherine_Tingley
  - 1918_Liga_Peruana_de_Football
  - Kaviyaloor
  - 1941_Dayton_Flyers_football_team
  - Kawanishi_K-3
  - 1948–49_Brentford_F.C._season
  - Kenneth_N._Good
  - 1950–51_Danish_1st_Division
  - Khalichian
  - 1951_Iowa_State_Teachers_Panthers_football_team
  - Khardun_Kola
  - 1956_South_Korean_local_elections
  - Khebda
  - 1964_Kansas_State_Wildcats_football_team
  - Kings_of_South_Beach
  - 1965_Colorado_State_Rams_football_team
  - Kino_MacGregor
  - 1968_New_Hampshire_Democratic_presidential_primary
  - Kintampo_Senior_High_School
  - 1969_Paddington_North_by-election
  - Kirk_Dunn

In [26]:
import pandas as pd

# Load Wikipedia CSV
df = pd.read_csv("results_conspiracy.csv")

# Strip any whitespace just in case
df.columns = df.columns.str.strip()

# Build a dictionary: model_name -> set of context_titles
model_titles = {
    model: set(group["context_title"]) 
    for model, group in df.groupby("model_name")
}

# Print summary of titles per model
for model, titles in model_titles.items():
    print(f"\nModel: {model} has {len(titles)} unique titles.")

# Compare across models
all_models = list(model_titles.keys())

# Pick first model as reference
ref_model = all_models[0]
ref_titles = model_titles[ref_model]

print(f"\nComparing other models to {ref_model}:")

for model in all_models[1:]:
    titles = model_titles[model]
    missing_in_model = ref_titles - titles   # titles in ref_model but missing here
    extra_in_model = titles - ref_titles     # titles in this model but not in ref_model
    print(f"\nModel: {model}")
    print(f"  Missing titles ({len(missing_in_model)}): {missing_in_model}")
    print(f"  Extra titles ({len(extra_in_model)}): {extra_in_model}")



Model: Qwen/Qwen3-8B has 1000 unique titles.

Model: google/gemma-2-9b has 1000 unique titles.

Model: meta-llama/Llama-3.1-8B has 1000 unique titles.

Model: mistralai/Mistral-7B-Instruct-v0.3 has 1000 unique titles.

Comparing other models to Qwen/Qwen3-8B:

Model: google/gemma-2-9b
  Missing titles (0): set()
  Extra titles (0): set()

Model: meta-llama/Llama-3.1-8B
  Missing titles (0): set()
  Extra titles (0): set()

Model: mistralai/Mistral-7B-Instruct-v0.3
  Missing titles (0): set()
  Extra titles (0): set()


In [ ]:
import pandas as pd

# Load Wikipedia CSV
df = pd.read_csv("results_conspiracy.csv")

# Strip whitespace from column names just in case
df.columns = df.columns.str.strip()

# Count rows per model
row_counts = df.groupby("model_name").size()

print("Number of rows per model:")
print(row_counts)


Number of rows per model:
model_name
Qwen/Qwen3-8B                         4016
google/gemma-2-9b                     2008
meta-llama/Llama-3.1-8B               2008
mistralai/Mistral-7B-Instruct-v0.3    2008
dtype: int64
